
Interactive review of Gemini OCR bounding box results

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import Image as IPImage, display, HTML
import os
from io import BytesIO
import cv2


In [28]:
# Setup project paths
script_dir = Path(__file__).parent if '__file__' in dir() else Path.cwd()
project_root = script_dir if (script_dir / "data").exists() else script_dir.parent

ocr_output_file = project_root / "data" / "02_raw_batch_mass" / "ocr_output.jsonl"
preprocessed_dir = project_root / "data" / "01_preprocessed"
preprocessed_meta_csv = preprocessed_dir / "all_metadata.csv"

print(f"Project Root: {project_root}")
print(f"OCR Output File: {ocr_output_file}")
print(f"Preprocessed Data Dir: {preprocessed_dir}")
print(f"\nFile exists: {ocr_output_file.exists()}")

Project Root: d:\Google Drive (smith318)\newspaper_ocr\newspaper_utilities
OCR Output File: d:\Google Drive (smith318)\newspaper_ocr\newspaper_utilities\data\02_raw_batch_mass\ocr_output.jsonl
Preprocessed Data Dir: d:\Google Drive (smith318)\newspaper_ocr\newspaper_utilities\data\01_preprocessed

File exists: True


In [29]:
meta_df = pd.read_csv(preprocessed_meta_csv)
print(f"Total metadata records loaded: {len(meta_df)}")
# meta_df['x_offset'] = pd.to_numeric(meta_df['x_offset'], errors='coerce')
# meta_df['y_offset'] = pd.to_numeric(meta_df['y_offset'], errors='coerce')
# print(type(meta_df['x_offset'][0]))
print(meta_df.head())

Total metadata records loaded: 3789
    pub_id        date  page_num  column  \
0  Alabama  0000-00-00         1       0   
1  Alabama  0000-00-00         1       1   
2  Alabama  0000-00-00         1       2   
3  Alabama  0000-00-00         2       0   
4  Alabama  0000-00-00         2       1   

                                                path  x_offset  y_offset  
0  data/01_preprocessed/Alabama/Alabama_p01_r00_c...         0      1533  
1  data/01_preprocessed/Alabama/Alabama_p01_r00_c...       867      1533  
2  data/01_preprocessed/Alabama/Alabama_p01_r00_c...      1547      1533  
3  data/01_preprocessed/Alabama/Alabama_p02_r00_c...         0       221  
4  data/01_preprocessed/Alabama/Alabama_p02_r00_c...       812       221  


In [30]:
# Load OCR output from JSONL
ocr_data = []
with open(ocr_output_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            ocr_data.append(json.loads(line))

df_ocr = pd.DataFrame(ocr_data)

print(f"Total OCR records loaded: {len(df_ocr)}")
print(f"\nDataframe shape: {df_ocr.shape}")
print(f"\nColumn names: {df_ocr.columns.tolist()}")
print(f"\nFirst few rows:")
print(df_ocr.head())

Total OCR records loaded: 345179

Dataframe shape: (345179, 9)

Column names: ['pub', 'page', 'col', 'text', 'conf', 'x', 'y', 'width', 'height']

First few rows:
       pub  page  col                                    text  conf    x   y  \
0  Alabama     2    0                  ALTOONA, 1,071, ETOWAH  0.99  245  24   
1  Alabama     2    0  KILPATRICK, LEWIS A. (b'86)-Ala.4,'09;  0.99  264  42   
2  Alabama     2    0                              (l'09); Ob  0.99  284  59   
3  Alabama     2    0      MOORE, DAVID SANDERS (b'52)⊕-Ga.5,  0.99  264  76   
4  Alabama     2    0                      '80; (l'80); R.D.2  0.99  284  93   

   width  height  
0    414    18.0  
1    586    18.0  
2    174    18.0  
3    586    18.0  
4    288    18.0  


In [31]:
df_ocr.describe()

,page,col,conf,x,y,width,height
count,345179.000000,345179.000000,345179.000000,345179.000000,345179.000000,345179.000000,345179.000000
mean,27.521115,1.000765,0.986446,123.280394,748.937065,412.374609,18.070505
std,27.519933,0.816694,0.006612,102.573092,456.614064,192.703643,2.823377
min,1.000000,0.000000,0.850000,13.000000,1.000000,9.000000,0.950000
25%,8.000000,0.000000,0.980000,54.000000,374.000000,284.000000,16.000000
50%,17.000000,1.000000,0.990000,73.000000,729.000000,428.000000,18.000000
75%,38.000000,2.000000,0.990000,195.000000,1093.000000,534.000000,19.000000
max,133.000000,2.000000,0.990000,710.000000,10019.000000,941.000000,48.000000


In [32]:
df_ocr['index_name'] = df_ocr.apply(lambda row: f"{row['pub']}_{row['page']}_{row['col']}", axis=1)
by_col = df_ocr.groupby('index_name')
by_col_groups = list(by_col.groups.keys())
print(f"\nTotal unique (pub, page, col) groups: {len(by_col_groups)}")
print(f"Example group keys: {by_col_groups[:5]}")


Total unique (pub, page, col) groups: 3789
Example group keys: ['Alabama_10_0', 'Alabama_10_1', 'Alabama_10_2', 'Alabama_11_0', 'Alabama_11_1']


In [33]:
# Helper function to find original image file
def find_original_image(state, page_num, col_idx):
    """
    Reconstruct the original image path from preprocessed directory.
    Images are stored as: data/01_preprocessed/{state}/{state}_p{page}_r00_c{col}.jpg
    """
    
    # Construct expected filename pattern
    img_path = preprocessed_dir / state /  f"{state}_p{page_num:02d}_r00_c{col_idx:03d}.jpg"
    if img_path.exists():
        return img_path
    
    return None

# Test the function
test_result = find_original_image(df_ocr.iloc[0]['pub'] if len(df_ocr) > 0 else 'Texas', 1, 0)
print(f"Test image search result: {test_result}")
print(f"Test path exists: {test_result.exists() if test_result else False}")

Test image search result: d:\Google Drive (smith318)\newspaper_ocr\newspaper_utilities\data\01_preprocessed\Alabama\Alabama_p01_r00_c000.jpg
Test path exists: True


In [34]:
# functions for interactions
def update_display(change=None):
    """Update display when item index changes"""
    idx = current_idx.value
    rows = by_col.get_group(by_col_groups[idx])
    
    col_info_display.value = by_col_groups[idx]
    context_display.value = f"""
    {rows[["text"]].to_html()}
    """
    
    # Display image
    img_path = find_original_image(rows.iloc[0]['pub'], rows.iloc[0]['page'], rows.iloc[0]['col'])
    # print(rows.iloc[0]['text'])
    if img_path:
        img = Image.open(img_path).convert("RGB")
        draw_obj = ImageDraw.Draw(img)
        # meta_row = meta_df.loc[
        #     (meta_df["pub_id"] == rows.iloc[0]['pub']) & 
        #     (meta_df["page_num"] == rows.iloc[0]['page']) & 
        #     (meta_df["column"] == rows.iloc[0]['col'])
        # ][["x_offset", "y_offset"]]
        print(f"Image size: {img.size}")
        max_x = max([row['x'] for _, row in rows.iterrows()])
        max_y = max([row['y'] for _, row in rows.iterrows()])
        wax_w = max([row['width'] for _, row in rows.iterrows()])
        print(f"max x: {max_x}, max y: {max_y}")
        details_display.value = f"Image size: {img.size}<br>max x: {max_x}, max y: {max_y}<br>max width: {wax_w}"
        for i, row in rows.iterrows():
            x=row['x'] #- meta_row['x_offset'].iloc[0]
            y=row['y'] #- meta_row['y_offset'].iloc[0]
            width=row['width']
            height=row['height']

        
            x=int(x / 1000 * img.width)
            y=int(y / max_y * img.height * .99)
            # width=int(width / 1000 * img.width)
            height=int(height / max_y * img.height)
            # draw_obj.rectangle([(100, 100), (img.width/2, img.height/2)], outline="red", width=6)
            # print((x, y), (y + row["width"], y + row["height"]))
            color = "red"
            if row['conf'] > 0.95:
                color = "orange"
            if row['conf'] >= 0.98:
                color = "yellow"
            if row['conf'] >= 0.99:
                color = "green"
            draw_obj.rectangle([(x, y), (x + width, y + height)], outline=color, width=2)
        with BytesIO() as f:
            img.save(f, format='PNG')
            img_bytes = f.getvalue()
            image_widget.value = img_bytes
            image_widget.width=img.width/2
            image_widget.height=img.height/2
            # image_widget = widgets.Image(
            #     value=img_bytes,
            #     format='png',
            #     width=img.width/2,
            #     height=img.height/2
            # )
 
def on_next(b):
    idx = current_idx.value
    if idx < len(by_col_groups) - 1:
        current_idx.value = idx + 1

def on_previous(b):
    if current_idx.value > 0:
        current_idx.value = current_idx.value - 1

In [35]:
# Create interactive review interface
current_idx = widgets.IntSlider(value=0, min=0, max=len(by_col_groups)-1, step=1, description='Item:', width='500px')
col_info_display = widgets.HTML(value='')
context_display = widgets.HTML(value='')
details_display = widgets.HTML(value='')

# display image
image_widget = widgets.Image()
   

current_idx.observe(update_display, names='value')

# Buttons for navigation
btn_next = widgets.Button(description='Next', button_style='info')
btn_previous = widgets.Button(description='Previous', button_style='warning')

btn_next.on_click(on_next)
btn_previous.on_click(on_previous)
buttons_box = widgets.HBox([btn_previous, btn_next])
left_box = widgets.VBox([col_info_display, details_display, buttons_box, ])

# Display the interface
update_display()
display(widgets.HBox([left_box, image_widget]))

Image size: (823, 2598)
max x: 274, max y: 1756
